# Slang measurement validation

This audit checks what the dictionary and contextual model detect on minimally cleaned CMV openings from the published **training** pairs. The sampling unit for detector precision and recall is one word position. The full-opening sample checks error in the proportion of words flagged.

## Blind annotation

1. Open `annotate_tokens.csv`. Label only the highlighted word in its sentence. If it belongs to a longer expression, label that word as part of the expression. Fill each field with 0 or 1. Keep the supplied item IDs.
2. Open `annotate_openings.csv` for the full-opening check. Count the numbered word positions belonging to **any** target category. Count each marked position once, even if it fits multiple categories. Enter the total in `target_word_count` and keep the item IDs.
3. Keep the scoring keys out of view during annotation. Save your completed files and set their paths in the scoring cell below.

| Label | What to mark |
| --- | --- |
| Slang | A nonstandard or community-specific expression or sense, such as “salty” meaning resentful. |
| Colloquial | Conversational wording, such as “gotta,” “gonna,” or “kinda.” |
| Internet shorthand | Online abbreviations and compact markers, such as “IMO,” “BTW,” or “TL;DR.” |
| Profanity | A swear word or a word participating in a profane expression. |
| Neutral | Set to 1 only when none of the four target labels apply in context. Ordinary names, acronyms, and literal uses usually belong here. |

The four target categories can overlap. Label the language used in context and leave judgments about argument quality out of the task. For `TL;DR`, count both word positions as shorthand when scoring the opening.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "team-setup").is_dir() and (p / "pyproject.toml").is_file()
)
RUNS = REPO / "project/data/processed/linguistic-features/slang-measurement-validation"
RUN_ID = None  # Set to the run whose CSVs you annotated if more than one run exists.
runs = sorted(RUNS.glob("*/design.json"))
design_path = RUNS / RUN_ID / "design.json" if RUN_ID else runs[-1]
run = design_path.parent
design = json.loads(design_path.read_text())
print(f"Annotation directory: {run}")
print(f"Independent full-opening sample: {design['opening_sample_size']} openings.")
strata = pd.DataFrame(design["strata"]).T
strata["Weight per item"] = strata["N_h"].div(strata["n_h"].replace(0, float("nan")))
display(strata.round(2))

Annotation directory: /Users/paolacalle/Desktop/NYU/semesters/fall-2026/ml-viz/ml-visualization-project/ml-visualization-project/project/data/processed/linguistic-features/slang-measurement-validation/20261120T212844469856Z
Independent full-opening sample: 30 openings.


,N_h,n_h,Weight per item
neither,1519652,100,15196.52
model_only,24345,75,324.60
dictionary_only,281,75,3.75
both,542,50,10.84


## Population estimates

The four strata use simple random samples without replacement. Tokens flagged by neither detector are sampled uniformly from that stratum. Each annotated token represents `N_h / n_h` population positions in the same stratum. We weight the confusion counts first, then calculate precision and recall. “Overall” means any of the four target categories. Category precision is the share of a detector’s flags that have that label. Category percentages can overlap.

Approximate 95% intervals apply shared random multipliers by OP discussion, calibrate each replicate to the known stratum sizes, and apply a finite-population correction within each stratum. Very rare categories and zero observed misses provide limited information. The scorer leaves intervals blank when the estimated proportion is at a boundary or its denominator is undefined.

The opening check reports mean absolute error and mean signed error in percentage points against hand-counted ratios. A positive signed error means the detector flagged a larger share than the annotator.

In [2]:
import sys

sys.path.insert(0, str(REPO / "project/notebooks/linguistic-features/informality"))
from validate_slang_measurement import LABELS, evaluate

TOKEN_LABELS = run / "annotate_tokens.csv"
OPENING_LABELS = run / "annotate_openings.csv"
token_complete = pd.read_csv(TOKEN_LABELS)[LABELS].notna().all().all()
opening_complete = pd.read_csv(OPENING_LABELS)["target_word_count"].notna().all()
detectors = {"dictionary": "Curated dictionary", "model": "Logistic regression"}
columns = [
    "Slang precision", "Shorthand precision", "Colloquial precision",
    "Profanity precision", "Overall target recall",
]
if token_complete:
    token_results, document_results = evaluate(
        run, TOKEN_LABELS, OPENING_LABELS if opening_complete else None,
    )
    precision = token_results.pivot(index="detector", columns="target", values="precision")
    summary = precision[["slang", "internet_shorthand", "colloquial", "profanity"]].copy()
    summary["overall_recall"] = token_results.loc[token_results["target"].eq("overall")].set_index(
        "detector"
    )["recall"]
    summary.columns = columns
    display(summary.rename(index=detectors).round(4))
    display(token_results.round(4))
    if document_results is not None:
        display(document_results.round(4))
    else:
        print("Opening error is pending complete opening counts.")
else:
    print("Population metrics are pending complete human token labels.")
    display(pd.DataFrame("TBD", index=list(detectors.values()), columns=columns))

Population metrics are pending complete human token labels.


,Slang precision,Shorthand precision,Colloquial precision,Profanity precision,Overall target recall
Curated dictionary,TBD,TBD,TBD,TBD,TBD
Logistic regression,TBD,TBD,TBD,TBD,TBD
